In [1]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)

sentence = "안녕하세요. 만나서 진심으로 반가워요."

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))

[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


In [2]:
# tfidf 를 사용한 벡터화
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = ["That movie is famous movie",
          "I like that actor",
          "I don't like that actor"]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [ ]:
# skipgram 모델 실습 베이직모델

import os
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings = vocab_size,
                                      embedding_dim = embedding_dim)
        self.linear = nn.Linear(in_features = embedding_dim,
                                out_features = vocab_size)
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [ ]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDT-17\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KD

In [5]:
# tokenizer 사용한 분해?

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [6]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens,
                    n_vocab = 5000,
                    special_tokens = ["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [7]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx + 1 : window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs
word_pairs = get_word_pairs(tokens, window_size = 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [8]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [10]:
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
word2vec = VanillaSkipgram(vocab_size = len(token_to_id),
                           embedding_dim = 128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

cuda


In [11]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 tensor(6.1965, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9820, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9327, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9028, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8808, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8631, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8485, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8356, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8240, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8137, device='cuda:0', grad_fn=<DivBackward0>)


In [12]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
tensor([ 1.7666, -0.0242,  0.2064, -0.2036, -0.3913,  0.0727,  0.0872, -0.0635,
         0.0956, -0.5047,  0.3042, -0.0328, -0.1894, -0.2769,  1.0485,  1.9836,
         0.3140,  0.8958,  0.0143, -0.6115,  1.3572,  0.2019, -0.5727, -0.1310,
        -1.8070,  0.9172,  0.0089, -0.0540,  1.3855,  1.6593,  1.8180, -0.6285,
        -1.0660,  0.2053,  0.0752,  1.2337, -0.6728,  0.3769,  0.4763, -0.2121,
         0.4269,  0.1928, -1.1449, -0.1496, -0.4740,  0.3701,  1.7771,  1.0306,
         0.1342, -0.6570,  0.8609, -0.0393,  0.0306, -1.1279,  0.1903,  0.3266,
         0.6291,  1.8278, -0.0340, -2.0264, -1.7316, -0.0441,  0.7067, -0.1673,
         1.6888, -0.2748, -0.5402,  0.6048, -0.2950,  1.2487,  0.7603, -1.0541,
        -0.9394,  0.0683, -0.4687, -0.7145,  0.2607,  1.3922,  0.2893, -0.3688,
         1.9642,  0.6985,  1.9347, -0.3776,  0.1848,  1.5579,  0.8197,  0.9936,
        -1.1548, -0.1094,  0.0421, -0.1333,  0.4982,  0.6267,  1.5886,  0.6976,
        -0.3931,  1.4144,  1.0436,  1

In [13]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b): # 두개의 단어를 넣으면 코사인유사도를 알려줌
    cosine = np.dot(b, a) / (norm(b, axis = 1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n): # 유사도에서 가장 가까운걸 보여주는
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1: n + 1]
    return top_n
cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n = 5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

전하 0.31770828
레알 0.28517666
두고 0.28496656
경험 0.26830328
성의 0.26788294


In [14]:
# Skipgram 보다 빠름 word2vector
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDT-17\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KD

In [15]:
# 형태소분석을 위한 Okt활용 토큰화

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

In [16]:
from gensim.models import Word2Vec

word2vec = Word2Vec(sentences = tokens,
                    vector_size = 128,
                    window = 5,
                    min_count = 1,
                    sg = 1,
                    epochs = 3,
                    max_final_vocab = 10000)

In [17]:
word2vec.save("models/word2vec.model")
word2vec = Word2Vec.load("models/word2vec.model")

In [19]:
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn = 5))
print(word2vec.wv.similarity(w1 = word, w2 = "연기력"))

[-4.74872440e-01 -4.61597815e-02  2.19536856e-01  3.16936553e-01
 -7.22375587e-02  1.89096108e-01 -4.51482050e-02 -1.05494998e-01
 -4.37131703e-01  3.90897542e-01 -2.66761724e-02 -2.93722749e-01
 -1.84462607e-01  1.85159609e-01  9.08549428e-02 -9.43520200e-03
 -2.15170816e-01  2.98925400e-01  6.19200356e-02  2.94483602e-01
  6.98721051e-01  3.82099122e-01 -1.59957394e-01  2.43506711e-02
 -2.42837280e-01 -2.86027268e-02 -3.00663561e-01  2.94474810e-01
 -2.91217454e-02 -1.99396506e-01 -4.60366935e-01  7.46400037e-04
  3.25664908e-01 -1.07588939e-01  1.35700136e-01  4.60364930e-02
 -1.19723566e-01  1.45666108e-01 -4.66850251e-01 -2.15457752e-01
 -3.24536394e-03  9.45065022e-02 -1.32819742e-01 -6.14241600e-01
 -1.94400549e-01  1.65194452e-01 -1.92298889e-01 -1.57408774e-01
  9.77831334e-02  3.93008813e-02  5.34291804e-01  5.50200939e-01
  1.11803755e-01  9.86420885e-02 -3.77963841e-01 -8.02987441e-02
 -1.15740143e-01  3.58170360e-01 -3.26882392e-01  2.05831215e-01
  1.13238511e-03 -1.63389

In [ ]:
# FastText 활용
from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

In [22]:
from gensim.models import FastText

fastText = FastText(sentences = tokens,
                    vector_size = 128,
                    window = 5,
                    min_count = 5,
                    sg = 1,
                    max_final_vocab = 20000,
                    epochs = 3,
                    min_n = 2,
                    max_n = 6)

In [23]:
oov_token = "사랑해요"
oov_vector = fastText.wv[oov_token]
print(oov_token in fastText.wv.index_to_key)
print(fastText.wv.most_similar(oov_vector, topn = 5))

False
[('사랑', 0.8846697211265564), ('사랑에', 0.8338861465454102), ('사랑의', 0.7898843288421631), ('사랑을', 0.758086085319519), ('사랑하는', 0.7376669049263)]


In [24]:
# 양방향 다층신경망을 활용한
# 문장을 가지고 RNN을 해보는것

import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [25]:
input_size = 128
output_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(input_size = input_size,
               hidden_size = output_size,
               num_layers = num_layers,
               nonlinearity = "tanh",
               batch_first = True,
               bidirectional = bidirectional).to(device)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size).to(device)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size).to(device)
outputs, hidden = model(inputs, h_0)

print(outputs.shape)
print(hidden.shape)
print(outputs.device)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])
cuda:0


In [26]:
# LSTM

input_size = 128
output_size = 256
num_layers = 3
bidirectional = True # 양방향에 장단기 메모리 보유
proj_size = 64

model = nn.LSTM(input_size = input_size,
                hidden_size = output_size,
                num_layers = num_layers,
                batch_first = True,
                bidirectional = bidirectional,
                proj_size = proj_size)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 proj_size if proj_size > 0 else output_size)

c_0 = torch.rand(num_layers * (int(bidirectional) + 1),
                 batch_size,
                 output_size)

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


c:\Users\KDT-17\Documents\17-NLP & CV\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1124: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\RNN.cpp:1474.)
  result = _VF.lstm(


In [27]:
# 문장분류 실습

import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout = 0.5,
                 bidirectional = True,
                 model_type = "lstm"):
        super().__init__()

        self.embedding = nn.Embedding(num_embeddings = n_vocab,
                                      embedding_dim = embedding_dim,
                                      padding_idx = 0)
        if model_type == "rnn":
            self.model = nn.RNN(input_size = embedding_dim,
                                hidden_size = hidden_dim,
                                num_layers = n_layers,
                                bidirectional = bidirectional,
                                dropout = dropout,
                                batch_first = True)
        elif model_type == "lstm":
            self.model = nn.LSTM(input_size = embedding_dim,
                                 hidden_size = hidden_dim,
                                 num_layers = n_layers,
                                 bidirectional = bidirectional,
                                 dropout = dropout,
                                 batch_first = True)

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [28]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\KDT-17\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\KD

In [30]:
train = corpus_df.sample(frac = 0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))

|       | text                                                                                     |   label |
|------:|:-----------------------------------------------------------------------------------------|--------:|
| 33553 | 모든 편견을 날려 버리는 가슴 따뜻한 영화. 로버트 드 니로, 필립 세이모어 호프만 영원하라. |       1 |
|  9427 | 무한 리메이크의 소재. 감독의 역량은 항상 그 자리에...                                    |       0 |
|   199 | 신날 것 없는 애니.                                                                       |       0 |
| 12447 | 잔잔 격동                                                                                |       1 |
| 39489 | 오랜만에 찾은 주말의 명화의 보석                                                         |       1 |
45000
5000


In [31]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens # ! . 같은거
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus = train_tokens,
                    n_vocab = 5000,
                    special_tokens = ["<pad>", "<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

In [33]:
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [[token_to_id.get(token, unk_id) for token in review] for review in train_tokens]
test_ids = [[token_to_id.get(token, unk_id) for token in review] for review in test_tokens]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


In [35]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype = torch.float32)
test_labels = torch.tensor(test.label.values, dtype = torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset,
                          batch_size = 16,
                          shuffle = True)
test_loader = DataLoader(test_dataset,
                         batch_size = 16,
                         shuffle = False)

print(len(train_loader))
print(len(test_loader))

2813
313


C:\Users\KDT-17\AppData\Local\Temp\ipykernel_26212\1475177882.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_ids = torch.tensor(train_ids)
C:\Users\KDT-17\AppData\Local\Temp\ipykernel_26212\1475177882.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_ids = torch.tensor(test_ids)


In [36]:
import torch.optim as optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

classifier = SentenceClassifier(n_vocab = n_vocab,
                                hidden_dim = hidden_dim,
                                embedding_dim = embedding_dim,
                                n_layers = n_layers).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr = 0.0001)

In [37]:
def train(model,
          datasets,
          criterion,
          optimizer,
          device,
          interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(step, np.mean(losses))

def test(model,
         datasets,
         criterion,
         device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits) > .5 # 긍정인지 부정인지 결과가 나와야함
        corrects.extend(torch.eq(yhat, labels).cpu().tolist())
    print(np.mean(losses), np.mean(corrects))

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier,
          train_loader,
          criterion,
          optimizer,
          device,
          interval)
    test(classifier,
         test_loader,
         criterion,
         device)

0 0.6818512082099915
500 0.6934097739988696
1000 0.6932513490542546
1500 0.6930771630498109
2000 0.6930650607578043
2500 0.6928991331476443
0.6921452780882009 0.5236
0 0.6820613145828247
500 0.6921622966102021
1000 0.6915131467324751
1500 0.6741974369277166
2000 0.6599270892345804
2500 0.6451110995587994
0.5663634256813854 0.7046
0 0.5674217939376831
500 0.5484482048157446
1000 0.5381217227353678
1500 0.5314883234439731
2000 0.5260013134508714
2500 0.5220209629130049
0.4941036829266685 0.7562
0 0.2859984040260315
500 0.4635269179077681
1000 0.472123524555555
1500 0.46991670245849476
2000 0.46901054723539215
2500 0.46519986644047634
0.46364108004128207 0.78
0 0.5231105089187622
500 0.43485517270431784
1000 0.43546169604752566
1500 0.43248185121778804
2000 0.42982636819193804
2500 0.4292373626006217
0.4503869558104311 0.7874


In [38]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 1.5203729  -0.90138346  0.19113809  0.24540873  0.30599013  1.0456609
 -0.36826152  0.32884607 -1.1991335  -0.03751028  0.02339854  1.2242424
  0.19331561 -1.0386783  -0.64703405 -0.5791855  -0.45844084 -1.1752007
 -1.754465   -1.214231    1.0354445   0.2532237  -0.99210405 -0.31852922
 -0.91066825 -0.05764545  0.9865785   0.04725919 -0.4916853  -0.3045039
 -1.0168048   0.31568432  0.3807211   0.11369513 -0.6706103   0.03204843
  2.2709882  -0.7321122  -0.12090988  1.4242201   1.6684542  -1.5551935
  1.7260133   0.7268495  -0.65912473  1.4184768   0.3870313  -1.1198845
  1.2237324   0.31184563 -0.3911231  -1.0763203  -0.28405023  0.8720266
  1.2170516   0.56524307 -2.1935308   0.17968118  1.9907885  -0.32019722
  0.4102979   0.2977473  -0.86037606 -0.44206545  1.4645374   0.90062314
  1.9976473  -0.8198917   1.8276119   1.2522241   0.4629659   1.29117
  0.99978244  1.3473549  -0.3400821   0.72933024  0.38619387 -1.8553958
 -0.45563337  0.10985447 -1.2916445  -0.649255    0.17632

In [ ]:
import torch.nn as nn

class SentenceClassifier(nn.Module):
    def __init__(self,
                 n_vocab,
                 hidden_dim,
                 embedding_dim,
                 n_layers,
                 dropout = 0.5,
                 bidirectional = True,
                 model_type = "lstm",
                 pretrained_embedding = None):
        super().__init__()

        if pretrained_embedding is not None:
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype = torch.float32))
        else:
            self.embedding = nn.Embedding(num_embeddings = n_vocab,
                                          embedding_dim = embedding_dim,
                                          padding_idx = 0)

        if model_type == "rnn":
            self.model = nn.RNN(input_size = embedding_dim,
                                hidden_size = hidden_dim,
                                num_layers = n_layers,
                                bidirectional = bidirectional,
                                dropout = dropout,
                                batch_first = True)
        elif model_type == "lstm":
            self.model = nn.LSTM(input_size = embedding_dim,
                                 hidden_size = hidden_dim,
                                 num_layers = n_layers,
                                 bidirectional = bidirectional,
                                 dropout = dropout,
                                 batch_first = True)

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)

        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

In [ ]:
# Skipgram 보다 빠름 word2vector
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

In [ ]:
train = corpus_df.sample(frac = 0.9, random_state = 42)
test = corpus_df.drop(train.index)

print(train.head().to_markdown())
print(len(train))
print(len(test))